# Download OpenBMI (Lee2019_MI)

OpenBMI is about 61 GB of MAT files for 54 subjects, too large to keep. Each subject is therefore downloaded
(~1.1 GB), preprocessed with the `openbmi` settings of `configs/datasets.yaml` (4-38 Hz band-pass,
common average reference unless `NO_CAR`, 100 Hz, 0-4 s after the cue), saved as `openbmi_SXX.npz`
(~20 MB, both sessions) and its raw files are deleted. The training runs only read these files.

Runs on Kaggle (no GPU needed, **Internet on**), on Colab, or on any machine.

1. Settings
2. Get the code
3. Python environment (Python 3.10 with the pinned packages)
4. Download and preprocess, `WORKERS` subjects at a time
5. Check the result

Afterwards on Kaggle: *Output -> New Dataset*, then attach that Dataset to `run.ipynb`
(existing: `openbmi-dataset-cuong` with CAR, `openbmi-nocar-cuong` without).
If the session ends early, run again with the earlier output visible (Kaggle: *Add Input -> Your Work ->
Notebook Output*): subjects already downloaded are copied over and skipped.

## 1. Settings

In [ ]:
NO_CAR = False          # True: no common average reference (for the no_car variant)
SUBJECTS = '1-54'       # e.g. '1-18' or '1,5,9'
WORKERS = 3             # subjects processed at the same time (each needs ~1.1 GB of disk, a few GB of RAM)
BRANCH = 'main'

## 2. Get the code

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path('/tmp/ssm-dan')                 # outside the saved output
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/CuongDM1806/ssm-dan.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
sys.path.insert(0, str(REPO))
from ssm_dan import launch
from ssm_dan.config import parse_subjects

platform = launch.detect()                  # kaggle | colab | local
OUT = platform.work / ('openbmi_cache_nocar' if NO_CAR else 'openbmi_cache')
OUT.mkdir(parents=True, exist_ok=True)
print('platform:', platform.name, '| output:', OUT)

## 3. Python environment

In [ ]:
PYTHON = launch.python_for_training(platform)   # creates a Python 3.10 environment once if needed
env = launch.environment(platform)              # raw MAT files go to a scratch directory, not the output
print('python:', PYTHON, '| raw downloads:', env['MNE_DATA'])

## 4. Download and preprocess

In [ ]:
import threading, time

subjects = parse_subjects(SUBJECTS, list(range(1, 55)))
todo = [s for s in subjects if not (OUT / f'openbmi_S{s:02d}.npz').exists()]
print(f'{len(subjects) - len(todo)} subjects already in {OUT}; downloading {todo}')

def download(worker, part):
    command = [PYTHON, '-u', '-m', 'ssm_dan.data.openbmi_cache', '--out', OUT,
               '--subjects', ','.join(map(str, part))] + (['--no-car'] if NO_CAR else [])
    code = launch._stream(command, OUT.parent / f'{OUT.name}_download_w{worker}.log', f'[w{worker}]', env)
    if code != 0:
        print(f'[w{worker}] stopped with exit code {code}; run again to finish its subjects')

threads = []
for worker in range(WORKERS):
    part = todo[worker::WORKERS]
    if part:
        threads.append(threading.Thread(target=download, args=(worker, part)))
        threads[-1].start()
        time.sleep(20)                          # stagger the MOABB set-up of the workers
for thread in threads:
    thread.join()

## 5. Check the result

In [ ]:
import json
import numpy as np

files = sorted(OUT.glob('openbmi_S*.npz'))
for path in files[:3]:
    with np.load(path) as data:
        print(path.name, '| session 1', data['X1'].shape, '| session 2', data['X2'].shape,
              '|', json.loads(str(data['signature'])))
missing = [s for s in subjects if not (OUT / f'openbmi_S{s:02d}.npz').exists()]
print(f'{len(files)} files, {sum(f.stat().st_size for f in files) / 1e9:.2f} GB in {OUT}')
print('All requested subjects are ready.' if not missing else
      f'NOT FINISHED: subjects {missing}. Run again with this output visible to finish them.')